# PLS with pyphi

PLS finds latent variables that maximise covariance between X and Y. Key outputs: scores T/U, X-loadings P, Y-loadings Q, weights W/Ws, VIP scores.

In [1]:
import pandas as pd
import numpy as np
import pyphi.calc as phi
import pyphi.plots as pp
from bokeh.io import output_notebook
output_notebook(hide_banner=True)
import pyphi.plots as _ppmod
_ppmod.output_file = lambda *a, **kw: None


Will be using the NEOS server in the absence of IPOPT and GAMS


## Load Data

In [2]:
features    = pd.read_excel('../data/Automobiles PLS.xlsx', 'Features',
                            na_values=np.nan, engine='openpyxl')
performance = pd.read_excel('../data/Automobiles PLS.xlsx', 'Performance',
                            na_values=np.nan, engine='openpyxl')
classid     = pd.read_excel('../data/Automobiles PLS.xlsx', 'CLASSID',
                            na_values=np.nan, engine='openpyxl')
print('X:', features.shape, '  Y:', performance.shape)


X: (406, 6)   Y: (406, 3)


## Build PLS Model

`cross_val_X=True` adds Q²X to assess X-predictability in addition to Q²Y.

In [3]:
plsobj = phi.pls(features, performance, 3, cross_val=5, cross_val_X=True)
print('Keys:', list(plsobj.keys()))
print('r2y (cumulative):', plsobj['r2y'])
print('q2Y (cross-val) :', plsobj['q2Y'])
print('T shape:', plsobj['T'].shape, '  Q shape:', plsobj['Q'].shape)


Cross validating LV #1


Cross validating LV #2


Cross validating LV #3


phi.pls using NIPALS and cross-validation (5%) executed on: 2026-03-26 23:43:27.059846
-------------------------------------------------------------------------------------------------------
PC #       Eig      R2X     sum(R2X)      Q2X     sum(Q2X)      R2Y     sum(R2Y)      Q2Y     sum(Q2Y)
PC #1:   3.850    0.776     0.776       0.775     0.775       0.541     0.541       0.538     0.538
PC #2:   0.143    0.042     0.818       0.065     0.840       0.136     0.677       0.095     0.632
PC #3:   0.750    0.151     0.969       0.128     0.968       0.023     0.700       0.065     0.698
-------------------------------------------------------------------------------------------------------
Keys: ['T', 'P', 'Q', 'W', 'Ws', 'U', 'r2x', 'r2xpv', 'mx', 'sx', 'r2y', 'r2ypv', 'my', 'sy', 'var_t', 'obsidX', 'varidX', 'obsidY', 'varidY', 'T2', 'T2_lim99', 'T2_lim95', 'speX', 'speX_lim99', 'speX_lim95', 'speY', 'speY_lim99', 'speY_lim95', 'q2Y', 'q2Ypv', 'q2X', 'q2Xpv', 'type']
r2y (cumulative):

## Variance Captured

In [4]:
pp.r2pv(plsobj)


## Score Plots

In [5]:
pp.score_scatter(plsobj, [1, 2], CLASSID=classid, colorby='Cylinders')
pp.score_scatter(plsobj, [1, 2], CLASSID=classid, colorby='Origin', add_ci=True)


## Loadings and Weights

In [6]:
pp.loadings(plsobj)
pp.weighted_loadings(plsobj)
pp.loadings_map(plsobj, [1, 2])


## VIP Scores

VIP > 1 suggests a variable is influential in the model.

In [7]:
pp.vip(plsobj)

## Predicted vs Observed

In [8]:
pp.predvsobs(plsobj, features, performance)
pp.predvsobs(plsobj, features, performance, CLASSID=classid, colorby='Origin')
pp.predvsobs(plsobj, features, performance, CLASSID=classid, colorby='Origin', x_space=True)


## Diagnostics

In [9]:
pp.diagnostics(plsobj, score_plot_xydim=[1, 2])


## Contribution Plots

In [10]:
pp.contributions_plot(plsobj, features, 'scores', to_obs=['Car1'])
pp.contributions_plot(plsobj, features, 'scores', to_obs=['Car1'], from_obs=['Car4'])
